<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания - 8
### Выполнил: Железняк Дамир Николаевич
### Группа: ПКТб-25-1


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----



Развитие системы управления кредитными картами (Спринт 1). Проект включает базовый класс CreditCard, инкапсулирующий номер карты (CardNumber), имя владельца (HolderName), срок действия (ExpiryDate) и баланс (Balance). Баланс защищен от установки отрицательных значений через геттер и сеттер. На основе базового класса реализованы три производных класса, демонстрирующие принципы наследования и полиморфизма: GoldCard (с бонусными милями), PremiumCard (с ассистентом поддержки) и CorporateCard (с привязкой к компании).

### Дополнительное задание:

Ко всем классам добавлены конструкторы, инициализирующие свойства. В базовый класс внедрен метод Transfer(CreditCard targetCard, decimal amount), который демонстрирует прямое взаимодействие между двумя объектами — безопасный перевод средств с текущей карты на целевую карту с проверкой доступного остатка.

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;

// --- 1. ИСПОЛНЯЕМЫЙ КОД ---

// Инициализация карт через конструкторы
GoldCard myGold = new GoldCard("1111-2222-3333-4444", "Мария Смирнова", "12/28", 10000m, 500);
PremiumCard myPremium = new PremiumCard("5555-6666-7777-8888", "Алексей Волков", "10/27", 5000m, "Анна");
CorporateCard myCorp = new CorporateCard("9999-0000-1111-2222", "Виктория Одинцова", "05/29", 50000m, "ООО 'Альфа-Софт'");

List<CreditCard> cards = new List<CreditCard> { myGold, myPremium, myCorp };

Console.WriteLine("=== ИСХОДНАЯ ИНФОРМАЦИЯ О КАРТАХ ===\n");
foreach (var card in cards)
{
    Console.WriteLine(card.GetInfo());
    card.CheckBalance();
    Console.WriteLine(new string('-', 50));
}

Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ ПОЛИМОРФИЗМА И ОПЛАТЫ ===\n");
myGold.Pay(1500m);     // Начислит мили
myPremium.CheckBalance(); // Предложит помощь ассистента, если потребуется
myCorp.Pay(20000m);   // Обычная оплата для корпоративной карты

Console.WriteLine("\n=== ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ (ПЕРЕВОД СРЕДСТВ) ===\n");
// Мария (Gold) переводит деньги Алексею (Premium)
myGold.Transfer(myPremium, 2500m);

// Попытка перевести больше, чем есть на балансе (Алексей пытается перевести 10000)
myPremium.Transfer(myCorp, 10000m);

Console.WriteLine("\n=== ИТОГОВЫЙ БАЛАНС ПОСЛЕ ОПЕРАЦИЙ ===\n");
foreach (var card in cards)
{
    card.CheckBalance();
}


// --- 2. ОБЪЯВЛЕНИЕ КЛАССОВ ---

// Базовый класс
public class CreditCard
{
    private decimal _balance;

    public string CardNumber { get; set; }
    public string HolderName { get; set; }
    public string ExpiryDate { get; set; }

    // Инкапсуляция атрибута Balance с логикой проверки
    public decimal Balance
    {
        get { return _balance; }
        set 
        { 
            if (value >= 0)
                _balance = value;
            else
                Console.WriteLine($"[Ошибка] Невозможно установить отрицательный баланс для карты {CardNumber}.");
        }
    }

    // Конструктор базового класса
    public CreditCard(string cardNumber, string holderName, string expiryDate, decimal initialBalance)
    {
        CardNumber = cardNumber;
        HolderName = holderName;
        ExpiryDate = expiryDate;
        Balance = initialBalance; // Используем свойство для проверки
    }

    public virtual string GetInfo()
    {
        return $"Карта: {CardNumber} | Владелец: {HolderName} | Годен до: {ExpiryDate}";
    }

    public virtual void Pay(decimal amount)
    {
        if (amount > 0 && Balance >= amount)
        {
            Balance -= amount;
            Console.WriteLine($"[Оплата] С карты {CardNumber} списано {amount:F2} руб. Текущий баланс: {Balance:F2} руб.");
        }
        else
        {
            Console.WriteLine($"[Отказ] Недостаточно средств на карте {CardNumber} для оплаты {amount:F2} руб.");
        }
    }

    public virtual void CheckBalance()
    {
        Console.WriteLine($"[Баланс] Доступный остаток по карте {CardNumber}: {Balance:F2} руб.");
    }

    // Метод, демонстрирующий взаимодействие объектов (перевод на другую карту)
    public void Transfer(CreditCard targetCard, decimal amount)
    {
        if (targetCard == null)
        {
            Console.WriteLine("[Ошибка] Целевая карта не найдена.");
            return;
        }

        if (amount > 0 && Balance >= amount)
        {
            Balance -= amount;
            targetCard.Balance += amount;
            Console.WriteLine($"[Перевод] Успешно переведено {amount:F2} руб. с карты {CardNumber} на карту {targetCard.CardNumber}.");
        }
        else
        {
            Console.WriteLine($"[Отказ перевода] Недостаточно средств на карте {CardNumber} для перевода {amount:F2} руб.");
        }
    }
}

// 1. Производный класс GoldCard
public class GoldCard : CreditCard
{
    public int BonusMiles { get; set; }

    // Конструктор с вызовом базового
    public GoldCard(string cardNumber, string holderName, string expiryDate, decimal initialBalance, int bonusMiles) 
        : base(cardNumber, holderName, expiryDate, initialBalance)
    {
        BonusMiles = bonusMiles;
    }

    public override void Pay(decimal amount)
    {
        // Проверяем, достаточно ли средств до вызова базового метода
        if (Balance >= amount) 
        {
            base.Pay(amount);
            int earnedMiles = (int)(amount / 100); 
            BonusMiles += earnedMiles;
            Console.WriteLine($"[Бонус] Начислено {earnedMiles} миль. Всего миль: {BonusMiles}.");
        }
        else
        {
            base.Pay(amount); // Базовый метод сам выведет сообщение об отказе
        }
    }
}

// 2. Производный класс PremiumCard
public class PremiumCard : CreditCard
{
    public string SupportAssistant { get; set; }

    // Конструктор с вызовом базового
    public PremiumCard(string cardNumber, string holderName, string expiryDate, decimal initialBalance, string supportAssistant) 
        : base(cardNumber, holderName, expiryDate, initialBalance)
    {
        SupportAssistant = supportAssistant;
    }

    public override void CheckBalance()
    {
        base.CheckBalance();
        Console.WriteLine($"[Поддержка] Персональный ассистент {SupportAssistant} на связи. Нужна помощь?");
    }
}

// 3. Производный класс CorporateCard
public class CorporateCard : CreditCard
{
    public string Company { get; set; }

    // Конструктор с вызовом базового
    public CorporateCard(string cardNumber, string holderName, string expiryDate, decimal initialBalance, string company) 
        : base(cardNumber, holderName, expiryDate, initialBalance)
    {
        Company = company;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $" | Компания: {Company}";
    }
}

Карта: 1111-2222-3333 | Владелец: Иван Иванов | Срок: 12.28
Текущий баланс: 50000 руб.

Карта: 7777-8888-9999 | Владелец: Петр Петров | Срок: 10.29 | Компания: ООО Инновации
Текущий баланс: 500000 руб.

Перевод 25000 руб. с карты 7777-8888-9999 на карту 1111-2222-3333 выполнен успешно.

Текущий баланс: 75000 руб.
Текущий баланс: 475000 руб.
